# 시즌 스탯 재생성 — `batter_stats_all_v2.xlsx` / `pitcher_stats_all_v2.xlsx`

드래프트 예측 모델(`scripts/export_predictions.py`, `모델/드래프트 예측 모델1.ipynb`)이
읽는 **선수-시즌 성적 파일 두 개**를 원본에서 다시 집계한다. 위에서부터 전부 실행하면 된다.

- **입력**: `season_full/full_tracked_{year}` (없으면 `season_full_{year}` → 경기별 원본 concat), `pitcher_box/pitcher_box_{year}.xlsx`
- **출력**: `model 및 배포/batter_stats_all_v2.xlsx`, `.../pitcher_stats_all_v2.xlsx`

## 주의 두 가지

1. **출력은 반드시 `model 및 배포` 폴더로 나간다.** `_drive_common` 이 그 폴더를 먼저
   뒤지므로, 루트에 저장하면 파이프라인은 낡은 파일을 계속 읽고 새로 만든 것은 조용히
   무시된다. 같은 이름 파일이 두 곳에 생기는 것 자체가 함정이다.
2. **한 해라도 실패하면 저장하지 않는다.** 예전 코드는 실패한 연도를 건너뛰고 저장해서,
   특정 시즌이 통째로 빠진 v2 가 조용히 만들어질 수 있었다. 부분 저장이 정말 필요하면
   설정 셀의 `ALLOW_PARTIAL = True` 로 명시적으로 켠다.

> 예전 탐색용 셀들(구 `batter_stats_all.xlsx` 읽기, 레이블 실험 등)은 `test.ipynb.bak_prev`
> 에 그대로 남겨뒀다. 이 노트북에는 실제로 v2 를 만드는 경로만 남긴다.

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [설정] 경로 · 연도 · stats 라이브러리
# ═══════════════════════════════════════════════════════════════
import sys, os, glob, re
import pandas as pd, numpy as np

BASE = '/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)'
SF   = f'{BASE}/season_full'                 # full_tracked / season_full
PBOX = f'{BASE}/pitcher_box'                 # pitcher_box_{year}.xlsx
LIB  = f'{BASE}/lib'                         # stats.py 사본
OUT  = f'{BASE}/model 및 배포'               # ★ 파이프라인이 읽는 폴더 — 여기에 저장한다

YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
ALLOW_PARTIAL = False        # True 로 바꾸면 일부 연도가 실패해도 저장한다(권장하지 않음)

if not os.path.exists(f'{LIB}/stats.py'):
    raise FileNotFoundError(
        f"{LIB}/stats.py 가 없습니다 - 저장소의 lib/stats.py 를 복사하세요")
sys.path.insert(0, LIB)
import stats

os.makedirs(OUT, exist_ok=True)
strip_no = lambda s: re.sub(r'\(\d+\)\s*$', '', str(s)).strip()   # "석지호(6)" → "석지호"

print(f'출력 폴더: {OUT}')
print(f'대상 연도: {YEARS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [로더] 그 해의 타석 단위(play-by-play) 데이터
#   타자: full_tracked 우선, 없으면 season_full, 그것도 없으면 경기별 원본 concat
#   NaN 정리를 여기서 끝낸다 — 예전엔 별도 패치 셀에서 load_pbp 를 감쌌는데,
#   그 셀을 안 돌리면 float NaN 이 파싱 함수에서 터졌다.
# ═══════════════════════════════════════════════════════════════
def load_pbp(year, verbose=True):
    df = None
    for name in [f'full_tracked_{year}.parquet', f'full_tracked_{year}.xlsx',
                 f'season_full_{year}.parquet', f'season_full_{year}.xlsx']:
        p = f'{SF}/{name}'
        if os.path.exists(p):
            df = pd.read_parquet(p) if name.endswith('.parquet') else pd.read_excel(p)
            if verbose: print(f'  ← {name} ({len(df):,}행)')
            break
    if df is None:
        files = sorted(glob.glob(f'{BASE}/{year}/*/*.xlsx')) or sorted(glob.glob(f'{BASE}/{year}/*.xlsx'))
        files = [f for f in files if '주자상황' not in f and '검수' not in f]
        if not files:
            raise FileNotFoundError(f'{year}: full_tracked도 season_full도 경기별 원본도 없음')
        if verbose: print(f'  ← 경기별 원본 {len(files)}개 concat 중...')
        df = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)
        if verbose: print(f'    {len(df):,}행')

    n0 = len(df)
    for c in ['result', 'batter', 'pitcher', 'team', 'def_team', 'details']:
        if c in df.columns:
            df[c] = df[c].fillna('').astype(str)
    if 'plate_seq' in df.columns:
        df['plate_seq'] = pd.to_numeric(df['plate_seq'], errors='coerce')
    df = df[(df['batter'].str.strip() != '') & (df['result'].str.strip() != '')]
    if verbose and len(df) != n0:
        print(f'  정리: {n0:,} → {len(df):,}행 (빈 batter/result 제거)')
    return df


print('연도별 소스 점검')
for y in YEARS:
    ft = any(os.path.exists(f'{SF}/full_tracked_{y}.{e}') for e in ['parquet','xlsx'])
    sf = any(os.path.exists(f'{SF}/season_full_{y}.{e}')  for e in ['parquet','xlsx'])
    n  = len(glob.glob(f'{BASE}/{y}/*/*.xlsx')) + len(glob.glob(f'{BASE}/{y}/*.xlsx'))
    pb = os.path.exists(f'{PBOX}/pitcher_box_{y}.xlsx')
    print(f'  {y}: full_tracked={"O" if ft else "-"} season_full={"O" if sf else "-"} '
          f'경기별={n}개  pitcher_box={"O" if pb else "-"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [빌더] 한 연도 → (타자 시즌스탯, 투수 시즌스탯)
#   계산 로직은 기존 그대로 — 지금 쓰이는 v2 와 같은 스키마를 유지해야
#   export_predictions.py 가 수정 없이 읽는다.
# ═══════════════════════════════════════════════════════════════
def build_year(year, verbose=True):
    if verbose: print(f'\n{"="*55}\n[{year}]')
    pbp = load_pbp(year, verbose)

    # 루상황/이벤트 플래그 — 루상황은 실패해도 기본 집계는 되므로 감싼다
    try:
        pbp = stats.add_context_columns(pbp)
    except Exception as e:
        if verbose: print(f'  ⚠ add_context_columns 실패(고급지표 생략): {e}')
    pbp = stats._add_event_flags(pbp)

    # ── 타자 ──
    B = stats.batting_totals(pbp, ['team', 'batter'])
    B['batter'] = B['batter'].map(strip_no)
    B = B.groupby(['team', 'batter'], as_index=False).sum(numeric_only=True)  # 등번호 변경 대비 재합산

    # 비율 재계산 (합산 후에는 다시 구해야 정확)
    ab, pa = B['AB'].where(B['AB'] > 0), B['PA'].where(B['PA'] > 0)
    B['AVG'] = B['H'] / ab
    if {'BB','HBP','SF'}.issubset(B.columns):
        B['OBP'] = (B['H'] + B['BB'] + B['HBP']) / (B['AB'] + B['BB'] + B['HBP'] + B['SF']).replace(0, np.nan)
    if {'2B','3B','HR'}.issubset(B.columns):
        B['SLG'] = (B['H'] + B['2B'] + 2*B['3B'] + 3*B['HR']) / ab
        B['OPS'] = B['OBP'] + B['SLG']
        B['IsoP'] = B['SLG'] - B['AVG']
    B['IsoD'] = B['OBP'] - B['AVG'] if 'OBP' in B else np.nan
    B['K%']   = B['SO'] / pa * 100 if 'SO' in B else np.nan
    B['BB%']  = B['BB'] / pa * 100 if 'BB' in B else np.nan

    # wOBA/wRC+ — 루상황이 살아 있을 때만
    try:
        _, _, ev = stats.compute_re24(pbp)
        w  = stats.compute_woba_weights(ev)
        la = stats.add_woba(stats.batting_totals(pbp, ['team','batter']), w)
        lg_pa, lg_r = la['PA'].sum(), la['R'].sum()
        lg_woba = (la['wOBA']*la['PA']).sum()/lg_pa
        B2 = stats.add_wrc(stats.add_woba(stats.batting_totals(pbp, ['team','batter']), w),
                           lg_woba, lg_r/lg_pa)
        B2['batter'] = B2['batter'].map(strip_no)
        B = B.merge(B2[['team','batter','wOBA','wRC','wRC+']].drop_duplicates(['team','batter']),
                    on=['team','batter'], how='left')
        evc = stats.re24_events_with_context(ev, pbp)
        col = 'batter_person_no' if 'batter_person_no' in evc.columns else 'batter'
        r24 = evc.groupby(col)['run_value'].sum()
        B['RE24'] = B['batter'].map({strip_no(k): v for k, v in r24.items()}) if col == 'batter' else np.nan
        if verbose: print('  wOBA/wRC+/RE24 계산됨')
    except Exception as e:
        if verbose: print(f'  ⚠ 고급지표 생략: {type(e).__name__}: {e}')
        for c in ['wOBA','wRC','wRC+','RE24']: B[c] = np.nan

    B['year'] = year
    B = B.rename(columns={'team': '학교', 'batter': '이름'})

    # ── 투수 ──
    pb = pd.read_excel(f'{PBOX}/pitcher_box_{year}.xlsx')
    pb['pitcher'] = pb['pitcher'].map(strip_no)
    P_ = stats.pitching_totals(pb, ['def_team', 'pitcher'])
    lg_ip = P_['IP'].sum()
    if lg_ip > 0 and {'ER','HR','BB_HBP','K'}.issubset(P_.columns):
        P_ = stats.add_fip(P_, stats.league_fip_constant(
            league_era=P_['ER'].sum()*9/lg_ip, league_hr=P_['HR'].sum(),
            league_bb_hbp=P_['BB_HBP'].sum(), league_k=P_['K'].sum(), league_ip=lg_ip))
    P_['BAA'] = P_['H'] / P_['AB'].where(P_['AB'] > 0)
    P_['year'] = year
    P_ = P_.rename(columns={'def_team': '학교', 'pitcher': '이름'})

    if verbose: print(f'  → 타자 {len(B)}명 / 투수 {len(P_)}명')
    return B, P_

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [실행 + 저장] 전 연도
#   ⚠ 오래 걸립니다 (경기별 concat 로 가는 연도는 특히)
# ═══════════════════════════════════════════════════════════════
bat_list, pit_list, failed = [], [], []
for y in YEARS:
    try:
        b, p = build_year(y)
        bat_list.append(b); pit_list.append(p)
    except Exception as e:
        failed.append((y, f'{type(e).__name__}: {e}'))
        print(f'  ❌ {y} 실패: {e}')

BAT = pd.concat(bat_list, ignore_index=True) if bat_list else pd.DataFrame()
PIT = pd.concat(pit_list, ignore_index=True) if pit_list else pd.DataFrame()

print(f'\n{"="*55}\n타자 {len(BAT):,}행 / 투수 {len(PIT):,}행')
print('\n[연도별]')
print(pd.DataFrame({'타자': BAT['year'].value_counts().sort_index(),
                    '투수': PIT['year'].value_counts().sort_index()}))
print('\n[고급지표 결측률]')
for c in ['wOBA','wRC+','RE24']:
    if c in BAT: print(f'  {c}: {BAT[c].isna().mean()*100:.1f}%')

# ── 저장 전 관문 ──────────────────────────────────────────────
# 실패한 연도가 있는데도 저장하면, 그 시즌이 통째로 빠진 v2 가 조용히 만들어지고
# 모델은 그걸 정상 데이터로 학습한다. 기본적으로 막는다.
if failed:
    print('\n실패한 연도:')
    for y, e in failed: print(f'  {y}: {e}')
    if not ALLOW_PARTIAL:
        raise RuntimeError(
            f'{len(failed)}개 연도가 실패해서 저장하지 않습니다 - 원인을 고치고 다시 실행하세요. '
            f'부분 저장이 정말 필요하면 설정 셀에서 ALLOW_PARTIAL = True 로 바꾸세요.')
    print('\n⚠ ALLOW_PARTIAL=True — 일부 연도가 빠진 채로 저장합니다.')

BAT_OUT = f'{OUT}/batter_stats_all_v2.xlsx'
PIT_OUT = f'{OUT}/pitcher_stats_all_v2.xlsx'
BAT.to_excel(BAT_OUT, index=False)
PIT.to_excel(PIT_OUT, index=False)
print(f'\n저장 완료\n  {BAT_OUT}\n  {PIT_OUT}')

# 검증 — 2020 드래프트 1라운드 선수가 잡히는가
print('\n[검증] 2020 드래프트 1라운드')
for nm in ['안재석','장재영','김건우','강효종','이의리','정민규']:
    h = pd.concat([BAT[BAT['이름']==nm][['year','이름','학교']],
                   PIT[PIT['이름']==nm][['year','이름','학교']]])
    print(f'  {nm}: {len(h)}행', h.values.tolist()[:4] if len(h) else '❌ 없음')